# 01 — Data Download and Cleaning

Part 1: loads the raw UCI Appliances Energy Prediction data (10-minute resolution, downloaded/cached on first run), cleans it, and resamples to hourly. See `src/appliance_energy/data.py` for the full logic — this notebook calls it directly and inspects the result.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
from IPython.display import Image, display

from appliance_energy.data import load_appliance_data
hourly = load_appliance_data(force_download=False)

Original (10-min) data shape: (19735, 28)
Date range: 2016-01-11 17:00:00 to 2016-05-27 18:00:00
Hourly data shape: (3290, 28)


## Resulting hourly series

In [2]:
print('Shape:', hourly.shape)
print('Date range:', hourly.index.min(), 'to', hourly.index.max())
print('Missing target values:', hourly['Appliances'].isna().sum())
hourly.head()

Shape: (3290, 28)
Date range: 2016-01-11 17:00:00 to 2016-05-27 18:00:00
Missing target values: 0


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,55.000000,35.000000,19.890000,46.502778,19.200000,44.626528,19.790000,44.897778,18.932778,45.738750,17.166667,55.116667,6.586667,84.260000,17.177778,41.400000,18.150000,48.710556,17.016667,45.446667,6.308333,733.750000,92.000000,6.166667,53.416667,5.050000,26.823044,26.823044
2016-01-11 18:00:00,176.666667,51.666667,19.897778,45.879028,19.268889,44.438889,19.770000,44.863333,18.908333,46.066667,17.111111,54.977778,6.180000,87.204444,17.229583,42.046806,18.094444,48.597222,16.981667,45.290000,5.941667,734.266667,91.583333,5.416667,40.000000,4.658333,22.324206,22.324206
2016-01-11 19:00:00,173.333333,25.000000,20.495556,52.805556,19.925556,46.061667,20.052222,47.227361,18.969444,47.815556,17.136111,55.869861,5.857361,88.131389,17.850000,45.017778,18.156111,49.213333,16.902222,45.311389,6.000000,734.791667,89.750000,6.000000,40.000000,4.391667,33.734932,33.734932
2016-01-11 20:00:00,125.000000,35.000000,20.961111,48.453333,20.251111,45.632639,20.213889,47.268889,19.190833,49.227917,17.615556,74.027778,5.469444,86.933889,17.632222,42.920000,18.773333,50.195556,16.890000,45.118889,6.000000,735.283333,87.583333,6.000000,40.000000,4.016667,25.679642,25.679642
2016-01-11 21:00:00,103.333333,23.333333,21.311667,45.768333,20.587778,44.961111,20.373333,46.164444,19.425556,47.918889,18.427222,69.037778,5.578889,86.129444,17.863611,43.618333,19.153333,49.542222,16.890000,44.807778,5.833333,735.566667,87.416667,6.000000,40.000000,3.816667,18.826274,18.826274


## Target variable summary

In [3]:
hourly['Appliances'].describe()

count    3290.000000
mean       97.779129
std        81.213695
min        28.333333
25%        50.000000
50%        63.333333
75%       110.000000
max       608.333333
Name: Appliances, dtype: float64

Resampling from 10-minute to hourly (mean), with small gaps filled by time interpolation, leaves 3,290 hourly observations spanning 11 Jan – 27 May 2016, with zero missing target values — a clean starting point for everything downstream.